<a href="https://colab.research.google.com/github/zbovaird/Agentic-Hemispheres/blob/main/colab_qwen25_3b_qlora_parser_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wolfram Guardrails Parser QLoRA Fine-Tune on Colab T4

This notebook runs the first small semantic-parser fine-tune for Wolfram Guardrails. It uses a Colab T4-friendly 3B model with QLoRA, saves the adapter and merged model, installs/pulls the matching Ollama 3B base model, and compares the tuned parser against the base parser on the reserved Cycle 6 holdout.

Recommended runtime: **GPU T4**.

North-star contract:

```text
prompt -> semantic JSON parser -> policy decision check
```

In Colab, comparison uses the repository's Python policy mirror because Wolfram Engine is usually not installed. After you bring the model back locally, run the Wolfram-backed comparison harness too.

## 1. Runtime Check

Use `Runtime -> Change runtime type -> T4 GPU` before running all cells.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

Tue Jun 23 18:02:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Configuration

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ZBovaird_glpay/wolfram-guardrails.git'
REPO_DIR = Path('/content/wolfram-guardrails')

HF_BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
OLLAMA_BASE_MODEL = 'qwen2.5:3b'
RUN_NAME = 'wolfram-guardrails-qwen25-3b-parser-v1'

TRAIN_FILE = REPO_DIR / 'results/finetune/splits/train.chat.jsonl'
VALIDATION_FILE = REPO_DIR / 'results/finetune/splits/validation.chat.jsonl'
TEST_FILE = REPO_DIR / 'results/finetune/splits/test.chat.jsonl'
CYCLE6_HOLDOUT = REPO_DIR / 'datasets/cycle6_fresh_blind_holdout_prompts.jsonl'

OUTPUT_ROOT = REPO_DIR / 'results/finetune/colab_qwen25_3b'
ADAPTER_DIR = OUTPUT_ROOT / 'adapter'
MERGED_DIR = OUTPUT_ROOT / 'merged_hf'
COMPARE_DIR = REPO_DIR / 'results/parser_model_comparison/colab_cycle6'

MAX_SEQ_LENGTH = 1024
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4

print('Repo:', REPO_URL)
print('HF base:', HF_BASE_MODEL)
print('Ollama base:', OLLAMA_BASE_MODEL)
print('Run name:', RUN_NAME)

Repo: https://github.com/ZBovaird_glpay/wolfram-guardrails.git
HF base: Qwen/Qwen2.5-3B-Instruct
Ollama base: qwen2.5:3b
Run name: wolfram-guardrails-qwen25-3b-parser-v1


## 3. Install Dependencies

This installs the fine-tuning stack and Ollama. The Ollama service is used for the base 3B comparison.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!pip -q install -U pip
!pip -q install -U \
  'transformers>=4.45.0' \
  'datasets>=2.20.0' \
  'accelerate>=0.33.0' \
  'peft>=0.12.0' \
  'trl>=0.9.6' \
  'bitsandbytes>=0.43.3' \
  sentencepiece \
  'protobuf>=5.29.1,<6.0.0' \
  safetensors
!curl -fsSL https://ollama.com/install.sh | sh
!python -m pip check || true

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.1 MB/s eta 0:00:00
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ip

## 4. Clone the GitHub Repository

### Check Repository Accessibility

Let's try to verify if the GitHub repository is accessible or if the URL is incorrect. We will use `requests` to check the URL's status and `git ls-remote` to check if Git can see the repository's references.

In [ ]:
import requests

try:
    response = requests.get(REPO_URL, timeout=10) # Set a timeout for the request
    print(f"HTTP Status Code for {REPO_URL}: {response.status_code}")
    if response.status_code == 200:
        print("The URL is reachable. This doesn't guarantee public access or repository existence, but confirms the server responded.")
    elif response.status_code == 404:
        print("The URL returned a 404 Not Found error. The repository might not exist at this address.")
    elif response.status_code == 403 or response.status_code == 401:
        print("The URL returned a 401/403 error. The repository might be private or require authentication.")
    else:
        print(f"Unexpected HTTP Status Code: {response.status_code}. Further investigation might be needed.")
except requests.exceptions.ConnectionError:
    print(f"Could not connect to {REPO_URL}. Check your internet connection or if the URL is valid.")
except requests.exceptions.Timeout:
    print(f"Request to {REPO_URL} timed out. The server might be slow or unreachable.")
except Exception as e:
    print(f"An error occurred while checking {REPO_URL}: {e}")

In [ ]:
import subprocess

try:
    # Try to list remote references without cloning
    print(f"Attempting to list remote references for {REPO_URL}...")
    output = subprocess.run(['git', 'ls-remote', REPO_URL], check=True, capture_output=True, text=True)
    print("git ls-remote successful. The repository likely exists and is accessible.")
    # print(output.stdout)
except subprocess.CalledProcessError as e:
    print(f"git ls-remote failed with exit status {e.returncode}.")
    print("This usually means the repository does not exist, is private, or you do not have permission to access it.")
    print("Error output:", e.stderr)
except FileNotFoundError:
    print("Git command not found. Make sure Git is installed in the environment.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## 5. Start Ollama and Pull the 3B Base Model

In [ ]:
import subprocess, time, os, signal

ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:3b
!ollama list

## 6. Load and Render the Chat Fine-Tune Data

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

raw = load_dataset('json', data_files={
    'train': str(TRAIN_FILE),
    'validation': str(VALIDATION_FILE),
    'test': str(TEST_FILE),
})

def render_chat(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(render_chat, remove_columns=raw['train'].column_names)
print(dataset)
print(dataset['train'][0]['text'][:800])

## 7. Train QLoRA Adapter

The settings are intentionally conservative for a T4: 4-bit NF4, batch size 1, gradient accumulation 8, and 3 epochs over 1,231 training rows. If Colab memory is tight, lower `MAX_SEQ_LENGTH` to 768.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_ROOT / 'trainer'),
    run_name=RUN_NAME,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Saved adapter to', ADAPTER_DIR)

## 8. Quick Adapter Smoke Test

In [ ]:
import json, torch

sample = json.loads(CYCLE6_HOLDOUT.read_text().splitlines()[0])
messages = [
    {'role': 'system', 'content': 'You are a semantic parser for Wolfram Guardrails. Return strict JSON only.'},
    {'role': 'user', 'content': sample['prompt']},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(trainer.model.device)
with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print('Prompt:', sample['prompt'])
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

## 9. Merge and Save the Fine-Tuned HF Model

This creates a normal Hugging Face model directory. If memory is tight, restart the runtime after saving the adapter, skip directly to this cell, and run it with the adapter already on disk.

In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
gc.collect()
torch.cuda.empty_cache()

base_for_merge = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_for_merge, str(ADAPTER_DIR)).merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(str(MERGED_DIR))
print('Saved merged model to', MERGED_DIR)

## 10. Compare Base Ollama 3B vs Tuned Parser on Cycle 6

Start with `--limit 30` for a fast sanity check. Remove the limit for the full 180-row holdout. Promotion still requires zero false allows on the full holdout.

In [ ]:
!python examples/compare_colab_qlora_parser.py \
  --dataset datasets/cycle6_fresh_blind_holdout_prompts.jsonl \
  --output-dir results/parser_model_comparison/colab_cycle6_smoke \
  --base-ollama-model qwen2.5:3b \
  --candidate-model results/finetune/colab_qwen25_3b/merged_hf \
  --limit 30

!cat results/parser_model_comparison/colab_cycle6_smoke/latest/summary.json

In [ ]:
# Full holdout comparison. This can take a while on a T4.
!python examples/compare_colab_qlora_parser.py \
  --dataset datasets/cycle6_fresh_blind_holdout_prompts.jsonl \
  --output-dir results/parser_model_comparison/colab_cycle6 \
  --base-ollama-model qwen2.5:3b \
  --candidate-model results/finetune/colab_qwen25_3b/merged_hf

!cat results/parser_model_comparison/colab_cycle6/latest/summary.json

## 11. Optional: Export Merged Model to Ollama in Colab

This creates a local Ollama model from the merged HF checkpoint using llama.cpp GGUF conversion. It is useful if you want to test the tuned model through Ollama before downloading it. It can take several minutes.

In [ ]:
# Optional cell. Run after the merged HF model exists.
import os, subprocess, textwrap
os.chdir('/content')
if not Path('/content/llama.cpp').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp.git', '/content/llama.cpp'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', '/content/llama.cpp/requirements.txt'], check=True)
gguf_f16 = OUTPUT_ROOT / f'{RUN_NAME}.f16.gguf'
subprocess.run(['python', '/content/llama.cpp/convert_hf_to_gguf.py', str(MERGED_DIR), '--outfile', str(gguf_f16), '--outtype', 'f16'], check=True)
modelfile = OUTPUT_ROOT / 'Modelfile'
modelfile.write_text(f'''FROM {gguf_f16}
PARAMETER temperature 0
SYSTEM You are a semantic parser for Wolfram Guardrails. Return strict JSON only.
''')
subprocess.run(['ollama', 'create', RUN_NAME, '-f', str(modelfile)], check=True)
subprocess.run(['ollama', 'list'], check=True)
os.chdir(REPO_DIR)

## 12. Download Artifacts

Download the adapter, merged HF model, and comparison results. The adapter is small; the merged model is larger.

In [ ]:
import shutil
os.chdir(REPO_DIR)
archive_base = '/content/wolfram_guardrails_qwen25_3b_parser_v1_artifacts'
shutil.make_archive(archive_base, 'zip', root_dir=str(OUTPUT_ROOT))
print(archive_base + '.zip')
print('Comparison results:', COMPARE_DIR)

## 13. Local Wolfram-Backed Evaluation After Download

Once the tuned model is available locally in Ollama, run the repository's Wolfram-backed comparison from your Mac:

```bash
python examples/compare_parser_models.py \
  --dataset datasets/cycle6_fresh_blind_holdout_prompts.jsonl \
  --output-dir results/parser_model_comparison/cycle6 \
  --base-model qwen2.5:3b \
  --candidate-model wolfram-guardrails-qwen25-3b-parser-v1
```

The Colab comparison is the training signal; the local Wolfram-backed comparison is the promotion gate.